```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a,A1b,A2,A3,A4a,A4b,A5a,A5b,A6a done;
    class A6b current;
    class A7,A8a,A8b,A9,A10,A11,A12 normal;
```

# Notebook 06b — Custom NER Tracking with Student Selection and EntityRuler

This notebook extends the outputs from **Notebook 05** (spaCy annotations) and **Notebook 06** (pretrained NER exploration) into a **student-driven custom tracking workflow**.

Instead of accepting all detected entities as equally useful, students will:

1. generate a **review file** of detected named entities grouped by label
2. manually choose which entities they want to track by changing a boolean from `False` to `True`
3. create a custom concept list in `./analysis/nb06-custom_concepts.txt`
4. build a **rule-based concept recognizer** with spaCy's `EntityRuler`
5. rerun NER analysis using:
   - their **selected named entities**
   - their **custom philosophical concepts**
6. visualize how these tracked entities and concepts vary across time bins

This notebook treats NER as a **research design decision**, not just a model output. Students curate what matters and then inspect the consequences analytically.


## Learning goals

By the end of this notebook, students should be able to:

- reuse annotation artifacts generated earlier in the workflow
- inspect and curate a detected entity inventory rather than blindly trusting model output
- build a human-in-the-loop entity selection file for downstream analysis
- create a custom philosophical concept lexicon
- add rule-based concept recognition with spaCy's `EntityRuler`
- combine pretrained NER with rule-based custom concept detection
- track selected entities and concepts across time bins
- reflect on how manual curation changes interpretive results


## Exercise structure

### Part A — Generate review files
The notebook will create:

- `analysis/tables/nb06_entity_selection_master.tsv`
- one optional TSV per entity label in `analysis/tables/nb06_entity_selection_by_label/`
- `analysis/nb06-custom_concepts.txt`

The selection files contain:

- `entity`
- `label`
- `selected` (default = `False`)
- frequency columns to help inspection

### Part B — Student editing task
Students must:

1. open the TSV selection file(s)
2. change `selected` from `False` to `True` for the named entities they want to track
3. keep `False` for noisy, irrelevant, or misleading entities
4. open `analysis/nb06-custom_concepts.txt`
5. replace the example lines with philosophical concepts they want to track

### Part C — Re-run analysis
After editing the files, students rerun the later sections of the notebook. The notebook will then:

- load the selected named entities
- build an `EntityRuler` from the custom concept list
- run custom concept recognition
- combine selected NER + custom concept matches
- visualize tracking results over time

> **Important methodological point:** the final analysis depends on student curation choices. Different choices may produce different historical stories.


In [1]:
# ------------------------------------------------------------
# Import
# ------------------------------------------------------------

from __future__ import annotations

from tqdm.auto import tqdm

from pathlib import Path
from collections import Counter, defaultdict
import re
import math

import numpy as np
import pandas as pd

import spacy
from spacy.tokens import DocBin, Span
from spacy.util import filter_spans

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
# ------------------------------------------------------------
# Paths and configuration
# ------------------------------------------------------------
PROJECT_ROOT = Path('.')
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'

ANALYSIS_DIR = PROJECT_ROOT / 'analysis'
FIGURES_DIR = ANALYSIS_DIR / 'figures'
TABLES_DIR = ANALYSIS_DIR / 'tables'
REPORTS_DIR = ANALYSIS_DIR / 'reports'
CACHE_DIR = PROJECT_ROOT / 'cache'

DOC_INDEX = TABLES_DIR / 'nb03-doc_index.csv'

# Split .spacy files from Notebook 05a
SPLIT_DIR = PROCESSED_DIR / 'nb05-corpus-split'

# Student-editable outputs
ENTITY_SELECTION_DIR = ANALYSIS_DIR / 'nb06-entity_selection'
ENTITY_SELECTION_DIR.mkdir(parents=True, exist_ok=True)
ENTITY_SELECTION_MASTER = ENTITY_SELECTION_DIR / 'nb06-entity_selection_master.tsv'
CUSTOM_CONCEPTS_PATH = ENTITY_SELECTION_DIR / 'nb06-custom_concepts.txt'

# NER / display controls
SPACY_MODEL = 'en_core_web_sm'
MIN_ENTITY_CHARS = 2
TOP_TRACKED_ITEMS = 20

print('DOC_INDEX:', DOC_INDEX)
print('SPACY_SPLIT:', SPLIT_DIR)
print('ENTITY_SELECTION_MASTER:', ENTITY_SELECTION_MASTER)
print('CUSTOM_CONCEPTS_PATH:', CUSTOM_CONCEPTS_PATH)


## Load the document index and the annotated corpus from Notebook 05

We reload the stable metadata table and the serialized spaCy `DocBin` produced earlier. This lets us preserve document IDs and time bins without reparsing the full corpus from scratch.


In [ ]:
# Load spaCy pipeline
print('Loading spaCy model:', SPACY_MODEL)
nlp = spacy.load(SPACY_MODEL)
print('Pipeline:', nlp.pipe_names)

In [ ]:
# Get all .spacy files in the directory
spacy_files = sorted(SPLIT_DIR.glob("*.spacy"))
print(f"\nFound {len(spacy_files)} split files.\n")

# Load all docs into a single list
all_docs = []
for filepath in spacy_files:
    docbin = DocBin().from_disk(filepath)
    docs = list(docbin.get_docs(nlp.vocab))
    all_docs.extend(docs)

print(f"\nLoaded {len(all_docs)} documents (chunks) total.")

## Reconstruct a document-level metadata table aligned with the loaded docs

Notebook 05 stored the corpus metadata in `doc.user_data`. We extract it here so entity matches can be linked back to document title, filename, and `time_bin`.


In [ ]:
doc_rows = []
for i, doc in enumerate(all_docs):
    doc_rows.append({
        'doc_i': i,
        'pg_id': doc.user_data.get('pg_id'),
        'title': doc.user_data.get('title'),
        'time_bin': doc.user_data.get('time_bin'),
        'n_tokens': len(doc),
    })

doc_meta = pd.DataFrame(doc_rows)

print(f"\nLoaded metadata for {len(doc_meta)} annotated documents' chunks.")
display(doc_meta.head())


## Store the documents' entities

In [ ]:
n_entities = sum(len(doc.ents) for doc in all_docs)
print('\nTotal stored entity spans:', f'{n_entities:,}', '\n')
print('Example entities from first doc:')
print([(ent.text, ent.label_) for ent in list(all_docs[0].ents)[:10]])

## Build a corpus-wide entity inventory

We aggregate all detected entities across the corpus and compute:

- total frequency
- document frequency
- number of time bins in which the entity appears

This creates a useful review table for student curation.


> The following function ensures that equivalent
    mentions like `social   contract` and `social contract` are treated as
    the same string in later analysis.

In [ ]:
def normalize_entity_text(text: str) -> str:
    """Normalize whitespace in an extracted entity string by
    replacing all runs of whitespace (spaces, tabs, newlines) with a single
    space and strips leading/trailing whitespace.
    """
    text = re.sub(r'\s+', ' ', str(text)).strip()
    return text

In [ ]:
entity_mentions = []

for doc_i, doc in enumerate(all_docs):
    meta = doc_meta.iloc[doc_i]

    for ent in doc.ents:
        ent_text = normalize_entity_text(ent.text)

        if len(ent_text) < MIN_ENTITY_CHARS:
            continue

        entity_mentions.append({
            'doc_i': doc_i,
            'pg_id': meta.get('pg_id'),
            'filename': meta.get('filename'),
            'title': meta.get('title'),
            'time_bin': meta.get('time_bin'),
            'entity': ent_text,
            'label': ent.label_,
        })

entities_df = pd.DataFrame(entity_mentions)

print('\nDetected entity mentions:', len(entities_df), '\n')
display(entities_df.head())


In [ ]:
entity_inventory = (
    entities_df
    .groupby(['label', 'entity'])
    .agg(
        mention_count=('entity', 'size'),
        doc_freq=('doc_i', 'nunique'),
        n_time_bins=('time_bin', lambda s: pd.Series(s).dropna().nunique()),
    )
    .reset_index()
    .sort_values(['label', 'mention_count', 'doc_freq', 'entity'], ascending=[True, False, False, True])
)

entity_inventory['selected'] = False

print('Unique entity-label pairs:', len(entity_inventory))
display(entity_inventory.head(20))


## Export custom selection files

This step creates the files that must edit.

### Next steps
1. Open the master TSV file or the per-label TSV files.
2. Change `selected` from `False` to `True` for the entities you want to track.
3. Save the file(s).
4. Then continue to the later sections of the notebook.

The notebook does **not** choose relevant entities automatically — that is part of the exercise.


In [ ]:
entity_inventory.to_csv(ENTITY_SELECTION_MASTER, sep='\t', index=False)

for label, subdf in entity_inventory.groupby('label'):
    safe_label = re.sub(r'[^A-Za-z0-9_\-]+', '_', str(label))
    out_path = ENTITY_SELECTION_DIR / f'nb06_entities_{safe_label}.tsv'
    subdf.sort_values(['mention_count', 'doc_freq', 'entity'], ascending=[False, False, True]).to_csv(
        out_path, sep='\t', index=False
    )

print('\nWrote master selection file:', ENTITY_SELECTION_MASTER)
print('Wrote per-label selection files to:', ENTITY_SELECTION_DIR)
print('Label files created:', entity_inventory['label'].nunique())


## Preview of the entity inventory by label

This is just a quick in-notebook view to support the editing task.


In [ ]:
TOP_ENTITIES_PER_LABEL_PREVIEW = 5

for label in sorted(entity_inventory['label'].unique()):
    print(f'\n### Label: {label}')
    display(
        entity_inventory.loc[entity_inventory['label'] == label]
        .head(TOP_ENTITIES_PER_LABEL_PREVIEW)
        .reset_index(drop=True)
    )

## Create the editable custom concept list for the EntityRuler

The notebook creates `analysis/nb06-custom_concepts.txt` if it does not already exist.

Students should edit the file so that it contains **one philosophical concept per line**.

Example entries:
- virtue
- practical reason
- natural law
- free will
- categorical imperative

Blank lines and lines starting with `#` are ignored.


In [ ]:
example_concepts = [
# Edit the list and add selected philosophical concepts or create the file and edit it
    'virtue',
    'reason',
    'free will',
    'natural law',
    'categorical imperative',
    'nature',
    'art',
    'state',
    'spirit',
    'time',
    'logic',
    'ethics'
    
]

if not CUSTOM_CONCEPTS_PATH.exists():
    CUSTOM_CONCEPTS_PATH.write_text('\n'.join(example_concepts) + '\n', encoding='utf-8')
    print('\nCreated starter concept file:', CUSTOM_CONCEPTS_PATH)
else:
    print('\nConcept file already exists:', CUSTOM_CONCEPTS_PATH)

print('\nCurrent file contents:')
print(CUSTOM_CONCEPTS_PATH.read_text(encoding='utf-8'))


# Fill the gap

Before continuing:

- edit `analysis/tables/nb06_entity_selection_master.tsv` **or** the per-label files
- edit `analysis/nb06-custom_concepts.txt`

Then rerun the cells below this point.

If you do not edit the files, the later analysis will either track nothing or only track the starter examples.


## Load the edited named-entity selections

We now read the master selection file and keep only rows where `selected == True`.

The parser below accepts common boolean spellings such as:
- `True`
- `TRUE`
- `1`
- `yes`
- `y`


In [ ]:
def coerce_bool(value) -> bool:
    if pd.isna(value):
        return False
    s = str(value).strip().lower()
    return s in {'true', '1', 'yes', 'y', 't'}

selected_df = pd.read_csv(ENTITY_SELECTION_MASTER, sep='\t')
selected_df['selected'] = selected_df['selected'].map(coerce_bool)

# Optional overlay:
# students may edit the master TSV, the per-label TSVs, or both.
# If per-label files were edited, we merge those choices back in.
overlay_frames = []

for fp in sorted(ENTITY_SELECTION_DIR.glob('nb06_entities_*.tsv')):
    sub = pd.read_csv(fp, sep='\t')

    if {'entity', 'label', 'selected'}.issubset(sub.columns):
        sub = sub[['entity', 'label', 'selected']].copy()
        sub['selected'] = sub['selected'].map(coerce_bool)
        overlay_frames.append(sub)

if overlay_frames:
    overlay_df = pd.concat(overlay_frames, ignore_index=True)
    overlay_df = overlay_df.drop_duplicates(subset=['entity', 'label'], keep='last')

    selected_df = (
        selected_df.drop(columns=['selected'])
        .merge(
            overlay_df,
            on=['entity', 'label'],
            how='left',
        )
    )

    selected_df['selected'] = selected_df['selected'].fillna(False)

selected_entities = selected_df.loc[selected_df['selected']].copy()

print('Selected named entities:', len(selected_entities))
display(selected_entities.head(20))


## Load the edited custom concept list

Each non-empty line becomes one rule-based concept pattern.


## Build a custom spaCy pipeline with `EntityRuler`

We add an `EntityRuler` for philosophical concepts and keep spaCy's pretrained `ner` component for standard named entities.

### Design choice
- custom concepts are labeled as `CONCEPT`
- we do **not** overwrite pretrained entities by default
- this means students can track:
  - curated named entities from the pretrained model
  - curated philosophical concepts from the rule-based layer


In [ ]:
# ------------------------------------------------------------
# Load custom concepts from the text file
# ------------------------------------------------------------
def load_concepts(path: Path) -> list[str]:
    concepts = []
    for line in path.read_text(encoding='utf-8').splitlines():
        item = line.strip()
        if not item or item.startswith('#'):
            continue
        concepts.append(item)
    return concepts

custom_concepts = load_concepts(CUSTOM_CONCEPTS_PATH)

print(f"\nLoaded {len(custom_concepts)} custom concepts from {CUSTOM_CONCEPTS_PATH}.\n")
print(custom_concepts[:25])

In [ ]:
# Lightweight pipeline: tokenizer + EntityRuler only
nlp_custom = spacy.blank("en")

ruler = nlp_custom.add_pipe(
    "entity_ruler",
    config={"phrase_matcher_attr": "LOWER"}
)

patterns = [{"label": "CONCEPT", "pattern": concept} for concept in custom_concepts]
ruler.add_patterns(patterns)

print(nlp_custom.pipe_names)

## Run the custom NER pipeline

This combines:

- pretrained named-entity detection
- rule-based concept recognition

We then extract only the items students actually chose to track.


In [ ]:
BATCH_SIZE = 2
custom_docs = []

with tqdm(total=len(all_docs), desc="Applying custom EntityRuler") as pbar:
    for i in range(0, len(all_docs), BATCH_SIZE):
        old_batch = all_docs[i:i + BATCH_SIZE]
        text_batch = [doc.text for doc in old_batch]

        new_batch = list(nlp_custom.pipe(text_batch, batch_size=BATCH_SIZE))

        for old_doc, new_doc in zip(old_batch, new_batch):
            new_doc.user_data = old_doc.user_data.copy()

            merged_ents = [Span(new_doc, ent.start, ent.end, label=ent.label_) for ent in old_doc.ents]
            merged_ents.extend(
                Span(new_doc, ent.start, ent.end, label=ent.label_) for ent in new_doc.ents
            )

            new_doc.ents = filter_spans(merged_ents)
            custom_docs.append(new_doc)

        pbar.update(len(old_batch))
print('\nCustom pipeline processed docs:', len(custom_docs))

### Apply named entities selection

In [ ]:
selected_lookup = set(
    zip(
        selected_entities['entity'].astype(str),
        selected_entities['label'].astype(str)
    )
)
custom_concept_lookup = set(custom_concepts)

tracked_rows = []

for doc_i, doc in enumerate(custom_docs):
    meta = doc_meta.iloc[doc_i]

    for ent in doc.ents:
        ent_text = normalize_entity_text(ent.text)
        ent_label = ent.label_

        is_selected_named_entity = (ent_text, ent_label) in selected_lookup
        is_selected_concept = (ent_label == 'CONCEPT' and ent_text in custom_concept_lookup)

        if not (is_selected_named_entity or is_selected_concept):
            continue

        tracked_rows.append({
            'doc_i': doc_i,
            'pg_id': meta.get('pg_id'),
            'title': meta.get('title'),
            'time_bin': meta.get('time_bin'),
            'entity': ent_text,
            'label': ent_label,
            'source': 'selected_ner' if is_selected_named_entity else 'entity_ruler',
        })

tracked_df = pd.DataFrame(tracked_rows)

if tracked_df.empty:
    print(
        'No tracked items found. Edit the selection TSV and/or the custom concepts file, ' 
        'then rerun the notebook from the loading step below the pause point.'
    )

else:
    print('\nTracked mentions after selection:', len(tracked_df))
    display(tracked_df.head(20))


## Quick coverage check

This section is intentionally diagnostic. If the table is empty, students should revisit their TSV selection file or concept list.


In [ ]:
coverage = (
    tracked_df.groupby(['source', 'label'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
display(coverage)

## Save the final tracked outputs

These files make the student-curated tracking results reusable in later notebooks or reports.


In [ ]:
TRACKED_MENTIONS_PATH = TABLES_DIR / 'nb06-custom_tracked_mentions.csv'
TRACKED_SUMMARY_PATH = TABLES_DIR / 'nb06-custom_tracked_summary.csv'

if not tracked_df.empty:
    tracked_df.to_csv(TRACKED_MENTIONS_PATH, index=False)

    tracked_summary = (
        tracked_df.groupby(['source', 'label', 'entity', 'time_bin'])
        .size()
        .reset_index(name='count')
        .sort_values(['source', 'label', 'count', 'entity'], ascending=[True, True, False, True])
    )
    tracked_summary.to_csv(TRACKED_SUMMARY_PATH, index=False)

    print('Saved tracked mentions:', TRACKED_MENTIONS_PATH)
    print('Saved tracked summary:', TRACKED_SUMMARY_PATH)
else:
    print('Nothing saved because tracked_df is empty.')


In [ ]:
tracked_summary.head()

In [ ]:
tracked_df.head()

## Visualize tracked items over time

These plots should now reflect **student choices** rather than the full raw NER output.


In [ ]:
# ------------------------------------------------------------
# Visualizations for custom NER tracking
# ------------------------------------------------------------
def bin_start(label) -> int:
    s = str(label)
    m = re.search(r'-?\d+', s.replace('–', '-'))
    return int(m.group(0)) if m else 10**9


def sort_time_bins(values) -> list[str]:
    vals = [str(v) for v in values if pd.notna(v)]
    return sorted(set(vals), key=bin_start) 


def save_df(df: pd.DataFrame, filename: str):
    out_path = TABLES_DIR / filename
    if filename.endswith('.tsv'):
        df.to_csv(out_path, sep='\t', index=False)
    else:
        df.to_csv(out_path, index=False)
    print('\nSaved table:', out_path, '\n')

In [ ]:
tracked_time = (
    tracked_df.groupby(['time_bin', 'source'])
    .size()
    .reset_index(name='count')
)

time_order = sorted(
    tracked_time['time_bin'].dropna().astype(str).unique().tolist(),
    key=bin_start
)

tracked_time['time_bin'] = pd.Categorical(
    tracked_time['time_bin'].astype(str),
    categories=time_order,
    ordered=True
)

tracked_time = tracked_time.sort_values(['time_bin', 'source'])

plt.figure(figsize=(12, 5))
sns.lineplot(
    data=tracked_time,
    x='time_bin',
    y='count',
    hue='source',
    marker='o'
)
plt.title('Tracked mentions over time by source')
plt.xlabel('Time bin')
plt.ylabel('Mention count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Mentions per label per time bin (mirrors tracked_time above, but grouped
# by label instead of source — this feeds the normalized plot below)
# ------------------------------------------------------------
label_time_mentions = (
    tracked_df.groupby(['time_bin', 'label'])
    .size()
    .reset_index(name='mention_count')
)

# Stable label order for consistent hues/legend ordering across plots
label_order = sorted(tracked_df['label'].dropna().unique().tolist())

print('Label/time_bin combinations:', len(label_time_mentions))
display(label_time_mentions.head())


In [ ]:
# ------------------------------------------------------------
# Optional normalization — mentions per tracked document in each time bin
# (useful when time bins have uneven corpus size)
# ------------------------------------------------------------
tracked_docs_per_bin = (
    tracked_df[['doc_i', 'time_bin']]
    .drop_duplicates()
    .groupby('time_bin')
    .size()
    .reset_index(name='n_tracked_docs')
)

normalized_label_time = label_time_mentions.merge(tracked_docs_per_bin, on='time_bin', how='left')
normalized_label_time['mentions_per_doc'] = (
    normalized_label_time['mention_count'] / normalized_label_time['n_tracked_docs'].replace(0, np.nan)
)
normalized_label_time['time_bin'] = pd.Categorical(
    normalized_label_time['time_bin'].astype(str), categories=time_order, ordered=True
)
normalized_label_time = normalized_label_time.sort_values(['time_bin', 'label'])
save_df(normalized_label_time, 'nb06b-label_time_mentions_normalized.csv')

plt.figure(figsize=(12, 6))
sns.lineplot(
    data=normalized_label_time,
    x='time_bin',
    y='mentions_per_doc',
    hue='label',
    hue_order=label_order,
    marker='o'
)
plt.title('Tracked entity mentions per tracked document over time')
plt.xlabel('Time bin')
plt.ylabel('Mentions per tracked document')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
fig_path = FIGURES_DIR / 'nb06b-label_mentions_per_doc_over_time.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved figure:', fig_path)


In [ ]:
# ----------------------------------------------------------------------
# Token totals by time bin
# ----------------------------------------------------------------------
doc_token_rows = []
for i, doc in enumerate(all_docs):
    doc_token_rows.append({
        "doc_i": i,
        "time_bin": doc.user_data.get("time_bin"),
        "n_tokens": len(doc),
    })

doc_tokens_df = pd.DataFrame(doc_token_rows)

tokens_by_bin = (
    doc_tokens_df.dropna(subset=["time_bin"])
    .groupby("time_bin", as_index=False)["n_tokens"]
    .sum()
    .rename(columns={"n_tokens": "total_tokens"})
)

# ----------------------------------------------------------------------
# 2) Raw counts for the two labels of interest
# ----------------------------------------------------------------------
focus_labels = ["PERSON", "CONCEPT", "ORG", "WORK_OF_ART"]

global_counts = (
    tracked_df[tracked_df["label"].isin(focus_labels)]
    .groupby(["time_bin", "label"], as_index=False)
    .size()
    .rename(columns={"size": "raw_count"})
)

# Ensure both labels appear in every time bin (fill missing with 0)
all_bins = sorted(tracked_df["time_bin"].dropna().astype(str).unique().tolist())
full_index = pd.MultiIndex.from_product(
    [all_bins, focus_labels],
    names=["time_bin", "label"]
)

global_counts = (
    global_counts.assign(time_bin=global_counts["time_bin"].astype(str))
    .set_index(["time_bin", "label"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

# ----------------------------------------------------------------------
# 3) Relative frequency per 1,000,000 tokens
# ----------------------------------------------------------------------
global_counts = global_counts.merge(tokens_by_bin.assign(time_bin=tokens_by_bin["time_bin"].astype(str)),
                                    on="time_bin", how="left")

global_counts["rel_freq"] = (
    global_counts["raw_count"] / global_counts["total_tokens"] * 1_000_000
)

# ----------------------------------------------------------------------
# 4) Plot raw vs relative
# ----------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

time_order = sorted(tokens_by_bin["time_bin"].unique().tolist(), key=bin_start)

# Raw counts
pivot_global_raw = global_counts.pivot(index="time_bin", columns="label", values="raw_count").reindex(time_order)
pivot_global_raw.plot(ax=ax1, marker="o", linestyle="-", linewidth=2, markersize=6)
ax1.set_title(f"Raw counts over time")
ax1.set_xlabel("Time Period")
ax1.set_ylabel("Raw Count")
ax1.legend(title="Label", bbox_to_anchor=(1.05, 1), loc="upper left")
ax1.grid(True, alpha=0.3)

# Relative frequency
pivot_global_rel = global_counts.pivot(index="time_bin", columns="label", values="rel_freq").reindex(time_order)
pivot_global_rel.plot(ax=ax2, marker="s", linestyle="--", linewidth=2, markersize=6)
ax2.set_title(f"Relative frequency (per 1M tokens)")
ax2.set_xlabel("Time Period")
ax2.set_ylabel("Frequency per 1,000,000 tokens")
ax2.legend(title="Label", bbox_to_anchor=(1.05, 1), loc="upper left")
ax2.grid(True, alpha=0.3)

for ax in (ax1, ax2):
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()

fig_path = FIGURES_DIR / "nb06b-total_label_counts.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure:", fig_path)
display(global_counts)

In [ ]:
# ------------------------------------------------------------
# Heatmap of top tracked items across time bins
# ------------------------------------------------------------
TOP_HEATMAP = 30

top_entity_names = (
    tracked_df['entity']
    .value_counts()
    .head(TOP_HEATMAP)
    .index
    .tolist()
)

heatmap_df = tracked_df[tracked_df['entity'].isin(top_entity_names)].copy()
heatmap_counts = (
    heatmap_df.groupby(['entity', 'time_bin'])
    .size()
    .reset_index(name='mention_count')
)
heatmap_counts['time_bin'] = pd.Categorical(
    heatmap_counts['time_bin'].astype(str), categories=time_order, ordered=True
)
heatmap_pivot = (
    heatmap_counts.pivot(index='entity', columns='time_bin', values='mention_count')
    .fillna(0)
)
heatmap_pivot = heatmap_pivot.loc[
    heatmap_pivot.sum(axis=1).sort_values(ascending=False).index
]
heatmap_pivot.to_csv(TABLES_DIR / 'nb06b_top_items_heatmap_matrix.csv')
print('Saved table:', TABLES_DIR / 'nb06b_top_items_heatmap_matrix.csv')

plt.figure(figsize=(12, max(6, TOP_HEATMAP * 0.45)))
sns.heatmap(heatmap_pivot, cmap='crest', linewidths=0.5)
plt.title(f'Top {TOP_HEATMAP} tracked items across time bins')
plt.xlabel('Time bin')
plt.ylabel('Tracked item')
plt.tight_layout()
fig_path = FIGURES_DIR / 'nb06b-top_items_heatmap.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved figure:', fig_path)

In [ ]:
# ------------------------------------------------------------
# Top tracked items overall per source
# ------------------------------------------------------------
top_items = (
    tracked_df.groupby(['entity', 'label', 'source'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

plt.figure(figsize=(12, 7))
sns.barplot(data=top_items, y='entity', x='count', hue='source')
plt.title(f'Top {TOP_TRACKED_ITEMS} tracked items')
plt.xlabel('Mention count')
plt.ylabel('Tracked entity / concept')
plt.tight_layout()
plt.show()

display(top_items)


In [ ]:
# ------------------------------------------------------------
# Top tracked items overall per label
# ------------------------------------------------------------
TOP_N = 20

top_items = (
    tracked_df.groupby(['label', 'entity'])
    .size()
    .reset_index(name='mention_count')
    .sort_values('mention_count', ascending=False)
    .head(TOP_N)
)
save_df(top_items, 'nb06b-top_tracked_items_overall_label.csv')
display(top_items)

plt.figure(figsize=(12, max(6, TOP_N * 0.35)))
sns.barplot(
    data=top_items,
    y='entity',
    x='mention_count',
    hue='label',
    dodge=False
)
plt.title(f'Top {TOP_N} tracked entities and concepts overall')
plt.xlabel('Mentions')
plt.ylabel('Tracked item')
plt.tight_layout()
fig_path = FIGURES_DIR / 'nb06b-top_tracked_items_overall_label.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved figure:', fig_path)

In [ ]:
# ------------------------------------------------------------
# Per-label top items over time (small multiples)
# (useful for comparing selected named entities with custom concepts)
# ------------------------------------------------------------
TOP_PER_LABEL = 5

label_item_counts = (
    tracked_df.groupby(['label', 'entity'])
    .size()
    .reset_index(name='mention_count')
)

top_per_label = (
    label_item_counts.sort_values(['label', 'mention_count', 'entity'], ascending=[True, False, True])
    .groupby('label', group_keys=False)
    .head(TOP_PER_LABEL)
)

focus_pairs = set(zip(top_per_label['label'], top_per_label['entity']))
facet_df = tracked_df[
    tracked_df.apply(lambda r: (r['label'], r['entity']) in focus_pairs, axis=1)
].copy()

facet_counts = (
    facet_df.groupby(['label', 'entity', 'time_bin'])
    .size()
    .reset_index(name='mention_count')
)
facet_counts['time_bin'] = pd.Categorical(
    facet_counts['time_bin'].astype(str), categories=time_order, ordered=True
)
facet_counts = facet_counts.sort_values(['label', 'entity', 'time_bin'])
save_df(facet_counts, 'nb06b-top_mentions_per_label_over_time.csv')

labels = sorted(facet_counts['label'].unique().tolist())
if labels:
    ncols = 2
    nrows = int(np.ceil(len(labels) / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(16, 6 * nrows), squeeze=False)

    for ax, label in zip(axes.flatten(), labels):
        sub = facet_counts[facet_counts['label'] == label]
        sns.lineplot(
            data=sub,
            x='time_bin',
            y='mention_count',
            hue='entity',
            marker='o',
            ax=ax
        )
        ax.set_title(f'Top tracked items over time — {label}')
        ax.set_xlabel('Time bin')
        ax.set_ylabel('Mentions')
        ax.tick_params(axis='x', rotation=45)
        ax.legend(title='Entity', fontsize=10, title_fontsize=9)

    for ax in axes.flatten()[len(labels):]:
        ax.axis('off')

    plt.tight_layout()
    fig_path = FIGURES_DIR / 'nb06b-top_mentions_per_label_over_time.png'
    plt.savefig(fig_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved figure:', fig_path)

In [ ]:
# ------------------------------------------------------------------------------
# Plot: Top tracked items over time for each label (Relative Frequency)
# ------------------------------------------------------------------------------

TOP_N_PER_LABEL = 5

# ----------------------------------------------------------------------
# 1) Token totals by time bin
# ----------------------------------------------------------------------
doc_token_rows = []
for doc in all_docs:
    doc_token_rows.append({
        "time_bin": doc.user_data.get("time_bin"),
        "n_tokens": len(doc),
    })

doc_tokens_df = pd.DataFrame(doc_token_rows)

tokens_by_bin = (
    doc_tokens_df.dropna(subset=["time_bin"])
    .assign(time_bin=lambda d: d["time_bin"].astype(str))
    .groupby("time_bin", as_index=False)["n_tokens"]
    .sum()
    .rename(columns={"n_tokens": "total_tokens"})
)

time_order = sorted(tokens_by_bin["time_bin"].unique().tolist(), key=bin_start)

# ----------------------------------------------------------------------
# 2) Raw counts by label, entity, and time bin
# ----------------------------------------------------------------------
item_counts = (
    tracked_df.dropna(subset=["time_bin", "label", "entity"])
    .assign(time_bin=lambda d: d["time_bin"].astype(str))
    .groupby(["label", "entity", "time_bin"], as_index=False)
    .size()
    .rename(columns={"size": "raw_count"})
)

# ----------------------------------------------------------------------
# 3) Select the top items overall within each label
# ----------------------------------------------------------------------
top_items = (
    item_counts.groupby(["label", "entity"], as_index=False)["raw_count"]
    .sum()
    .sort_values(["label", "raw_count", "entity"], ascending=[True, False, True])
    .groupby("label", group_keys=False)
    .head(TOP_N_PER_LABEL)
)

plot_df = item_counts.merge(
    top_items[["label", "entity"]],
    on=["label", "entity"],
    how="inner"
)

# ----------------------------------------------------------------------
# 4) Ensure every selected item has a row for every time bin
# ----------------------------------------------------------------------
all_pairs = top_items[["label", "entity"]].drop_duplicates()
full_index = pd.MultiIndex.from_product(
    [
        all_pairs["label"].unique(),
        all_pairs["entity"].unique(),
        time_order,
    ],
    names=["label", "entity", "time_bin"]
)

plot_df = (
    plot_df.set_index(["label", "entity", "time_bin"])
    .reindex(full_index)
    .reset_index()
)

# Keep only valid selected (label, entity_norm) pairs
valid_pairs = set(map(tuple, all_pairs[["label", "entity"]].to_records(index=False)))
plot_df = plot_df[
    plot_df[["label", "entity"]].apply(tuple, axis=1).isin(valid_pairs)
].copy()

plot_df["raw_count"] = plot_df["raw_count"].fillna(0)

# ----------------------------------------------------------------------
# 5) Add relative frequency per 1M tokens
# ----------------------------------------------------------------------
plot_df = plot_df.merge(tokens_by_bin, on="time_bin", how="left")

plot_df["rel_freq"] = (
    plot_df["raw_count"] / plot_df["total_tokens"] * 1_000_000
)

# ----------------------------------------------------------------------
# 6) Plot one panel per label
# ----------------------------------------------------------------------
labels = sorted(plot_df["label"].dropna().unique().tolist())
n_labels = len(labels)

ncols = 2
nrows = math.ceil(n_labels / ncols)

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(16, max(5 * nrows, 6)),
    squeeze=False
)

for ax, label in zip(axes.flat, labels):
    label_df = plot_df[plot_df["label"] == label].copy()

    pivot_df = label_df.pivot(
        index="time_bin",
        columns="entity",
        values="rel_freq"
    ).reindex(time_order)

    pivot_df.plot(
        ax=ax,
        marker="o",
        linestyle="-",
        linewidth=2,
        markersize=5
    )

    ax.set_title(f"Top {TOP_N_PER_LABEL} items for {label} over time")
    ax.set_xlabel("Time Period")
    ax.set_ylabel("Frequency per 1,000,000 tokens")
    ax.grid(True, alpha=0.3)
    ax.legend(title="Item", bbox_to_anchor=(1.05, 1), loc="upper left")
    ax.tick_params(axis='x', rotation=45)

# Hide any unused subplot axes
for ax in axes.flat[n_labels:]:
    ax.set_visible(False)

plt.tight_layout()

fig_path = FIGURES_DIR / "nb06b-top_freq_per_label_rel.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure:", fig_path)
display(top_items.sort_values(["label", "raw_count"], ascending=[True, False]))

## Representative document lookup for tracked items

Counts alone are not enough. We should also inspect where tracked items occur.


In [ ]:
tracked_doc_counts = (
    tracked_df.groupby(['entity', 'label', 'title', 'time_bin'])
    .size()
    .reset_index(name='count')
    .sort_values(['count', 'entity'], ascending=[True, True])
)

display(tracked_doc_counts.tail(30))


In [ ]:
# ------------------------------------------------------------
# Basic summary tables
# ------------------------------------------------------------
print('Tracked entity mentions:', f'{len(tracked_df):,}')
print('Tracked labels:', sorted(tracked_df['label'].dropna().unique().tolist()))
print('Tracked docs:', tracked_df['doc_i'].nunique())
print('Tracked time bins:', sort_time_bins(tracked_df['time_bin'].dropna().unique()))

overall_item_counts = (
    tracked_df.groupby(['label', 'entity'])
    .size()
    .reset_index(name='mention_count')
    .sort_values(['mention_count', 'label'], ascending=[False, True])
)
save_df(overall_item_counts, 'nb06b-tracked_entity_counts.csv')
display(overall_item_counts.head(20))



## Reflection questions

1. Which pretrained named entities did you keep, and which did you exclude?
2. Did the pretrained NER produce false positives that looked plausible at first glance?
3. Which philosophical concepts worked well with the `EntityRuler`?
4. Which concepts were difficult because of ambiguity, spelling variation, or historical phrasing?
5. How did your manual selections change the apparent temporal story?
6. What is gained, and what is lost, when moving from automatic detection to student-curated tracking?


 =============================================== YOUR THOUGHTS HERE ===============================================



---

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A6b highlight;
```